In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from tqdm import tqdm


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# Add sentencizer safely
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving filtered_abstracts_cancer.csv to filtered_abstracts_cancer.csv


In [ ]:
df= pd.read_csv('filtered_abstracts_cancer.csv')

In [ ]:
df.columns

Index(['PMID', 'Title', 'Abstract', 'Authors', 'Affiliation', 'Country',
       'Journal', 'Year', 'DOI', 'URL', 'filtered_abstract'],
      dtype='object')

In [ ]:
df = df.drop_duplicates(subset=["filtered_abstract"])
df = df[df["filtered_abstract"].notna()]
df = df[df["filtered_abstract"].str.strip() != ""]


In [ ]:
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()

    text = re.sub(r"\s+", " ", text)

    text = re.sub(r"[^\w\s\.\,\-\(\)/%]", " ", text)

    text = re.sub(r"([.,!?]){2,}", r"\1", text)

    return text.strip()

df["filtered_abstract"] = df["filtered_abstract"].apply(clean_text)


In [ ]:
def preserve_entity_punctuation(text):
    return re.sub(r"(?<!\d)\.(?!\d)", " ", text)

df["filtered_abstract"] = df["filtered_abstract"].apply(preserve_entity_punctuation)


In [ ]:
df["filtered_abstract"] = df["filtered_abstract"].str.lower()

In [ ]:
def sentence_split(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

tqdm.pandas(desc="Sentence segmentation")
df["sentences"] = df["filtered_abstract"].progress_apply(sentence_split)

df = df.explode("sentences").reset_index(drop=True)


Sentence segmentation: 100%|██████████| 26700/26700 [04:30<00:00, 98.86it/s] 


In [ ]:
def tokenize(sentence):
    return re.findall(r"\b[\w\.\-/%]+\b", sentence)

df["tokens"] = df["sentences"].apply(tokenize)

In [ ]:
def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS]

df["tokens"] = df["tokens"].apply(remove_stopwords)


In [ ]:
def lemmatize_tokens(tokens):
    doc = nlp(" ".join(tokens))
    return [
        token.lemma_
        for token in doc
        if not token.is_punct and not token.is_space
    ]

tqdm.pandas(desc="Lemmatization")
df["final_tokens"] = df["tokens"].progress_apply(lemmatize_tokens)


Lemmatization: 100%|██████████| 32749/32749 [03:22<00:00, 161.61it/s]


In [ ]:
output_path = "preprocessed_cancer.csv"

df.to_csv(output_path, index=False)

print(f"✅ Dataset saved to {output_path}")


✅ Dataset saved to preprocessed_cancer.csv


In [ ]:
from google.colab import files

files.download("preprocessed_cancer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>